# GCM BA - 3. Apply F1 to EC-Earth3 historical and to the four GCM runs

The **same** F1 (notebook 2, EC-Earth3 grid) is applied unchanged to:
EC-Earth3 historical (for the verification), 2025 original, 2025 hosing, 2050 original, 2050 hosing.
Result: **"ERA5-like" GCM runs on the EC-Earth3 grid (0.70 deg)**, over Europe (`REGION` of notebook 2: RCM + LISFLOOD
domains + 1 deg). The 2025 original one is the
reference for the bias adjustment of the RCM. `rsds` is not bias-adjusted.

**Method (Cannon et al. 2015, version tag `bu.F1_VARIANT`, set in `ba_utils.py`)**:
- each calendar month m is calibrated on the days of the **3 months centred on m** (`CAL_WINDOW_MONTHS = 3`, "sliding
  3-month windows centered on the month of interest") and applied to month m only: consecutive months share 2/3 of
  their calibration data, which reduces the jumps of the correction at the changes of month;
- **pr** (`PR_METHOD = "jitter"`): values below 0.1 mm/day are replaced by a uniform noise in (0, 0.1) in ERA5, in
  EC-Earth3 historical and in the runs, the multiplicative QDM is applied to **all days**, and values below 0.1 mm/day
  are set back to 0 afterwards. The quantile mapping itself then corrects the wet-day frequency, and the model's change
  in dry-day frequency is carried by QDM (`PR_METHOD = "wdf"`: previous fixed WDF + QDM on wet days only). Threshold
  0.1 mm/day: GloFAS was calibrated with ERA5, so its light-precipitation days are kept;
- 100 quantiles, linear interpolation between them; pr above the model's last historical quantile: plain QDM (the
  factor of the last quantile, xsdba constant extrapolation), retained after the test of `0_precip_regimes`.

Application with xsdba (`QuantileDeltaMapping.adjust`, `extrapolation="constant"`, `INTERP`), month by month with that
month's calibration: each value is ranked within its month (whole run, the runs being quasi-stationary) and gets the
correction of that rank. **pr above the model's historical range** (`PR_TAIL = None`): plain QDM, a value above the
model's last historical quantile hq (per pixel and month) is multiplied by the factor of the last quantile (ERA5's last
quantile / hq), as in Cannon et al. (2015). Tested over Europe in `0_precip_regimes` section 4 against the additive rule
`QDM(hq) + (x - hq)` (Boe et al. 2007, `PR_TAIL = "additive"`) and a targeted cap (method note, Section 5.2): no
outliers (max 295 mm/day), best Rx1day calibration, change of Rx1day preserved (slope 0.95-1.00 vs 0.84-0.93 with the
additive rule). The 800-1300 mm/day values of the first version are in dry regions outside Europe, where the outputs
are not used. Computing time: keep `USE_DASK_CLUSTER = False` (default threaded
scheduler); in a local test a dask cluster of processes was ~8x slower for this step.

One file per run: `GCM_bias_adjusted_0.7deg/<run>_F1_<period>_<variant>.nc` (e.g. `gcm_2025_hosing_F1_1981-2014_w3m-jitter_europe.nc`).

```
elisa/GCM_BA/                           (EOS; inputs read directly from GCM_downscaling/Input_data/ and EC-Earth_hosing/)
├── 2_F1_calculation/    F1_<period>/       : WDF rule + QDM parameters
├── GCM_bias_adjusted_0.7deg/  <run>_F1_<period>.nc : historical + 4 GCM runs bias-adjusted (ERA5-like), EC-Earth3 grid
└── 4_verification/cache/                   : statistics of the verification
```

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_CORE = cfg["variables"]                                   # pr, tasmax, tasmin: required
VARS_ALL = VARS_CORE                # rsds not bias-adjusted (to add it back: VARS_CORE + cfg.get("extra_variables", []))
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]

CAL_PERIOD = ("1981", "2014")      # years of ERA5 / EC-Earth3 historical used for F1 (EC-Earth3 historical ends in 2014)
RSDS_FIRST_YEAR = "2005"           # ERA5 radiation starts in 2005: shorter period for rsds

def period_of(v):
    """Calibration period of a variable: CAL_PERIOD, starting in 2005 for rsds."""
    return (max(CAL_PERIOD[0], RSDS_FIRST_YEAR), CAL_PERIOD[1]) if v == "rsds" else CAL_PERIOD

REGION_MARGIN = 1.0                # degrees added around the RCM and LISFLOOD domains
REGION = bu.gcm_ba_region(cfg, REGION_MARGIN)   # Europe: box of the RCM + LISFLOOD domains (None = the whole globe)
F1_TAG = (f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}"   # F1 version (ba_utils: CAL_WINDOW_MONTHS, PR_METHOD)
          + ("_europe" if REGION else ""))
print("region:", REGION)
CHUNKS_SPACE = {"lat": 32, "lon": 64}
INTERP = "linear"        # xsdba QDM interpolation between quantiles: "linear" (Cannon et al. 2015) | "nearest"
PR_TAIL = None           # pr above the model's historical range: None = plain QDM (retained, 0_precip_regimes section 4)
                         # | "additive" = QDM(hq) + x - hq (Boe et al. 2007)
OVERWRITE = False
F1_DIR = bu.gcm_ba_dir(cfg, "2_F1_calculation", F1_TAG)
OUT = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")      # bias-adjusted GCM runs (0.7 deg)
assert (F1_DIR / "qdm_pr.nc").exists(), "F1 not calibrated: run notebook 2 first."
RUNS = ["gcm_historical"] + GCM_RUNS
RUNS_TO_DO = None        # None = every run | e.g. ["gcm_2025_hosing"]: open several copies of this notebook, one run each,
                         # to compute the runs in parallel (start with ["gcm_historical"], the shortest, to time it)


def run_input(name, v):
    """Raw input of a run (EC-Earth3 grid), standardised when read; historical over the calibration period."""
    return bu.gcm_ba_input(cfg, name, v, period_of(v) if name == "gcm_historical" else None, REGION)
print("F1:", F1_DIR, "| outputs:", OUT)

## 1. Application

In [ ]:
import time
for name in (RUNS if RUNS_TO_DO is None else RUNS_TO_DO):
    out_path = OUT / f"{name}_{F1_TAG}.nc"
    if out_path.exists() and not OVERWRITE:
        print(f"[already done] {name}"); continue
    vs = [v for v in VARS_ALL if bu.gcm_ba_available(cfg, name, v)]
    if not all(v in vs for v in VARS_CORE):
        print(f"[missing] {name}: inputs missing (config, notebook 00)"); continue
    print(f"-> {name}: {', '.join(vs)}")
    sim = {v: run_input(name, v) for v in vs}
    t0 = time.time()
    adj = bu.apply_F(sim, F1_DIR, chunks_space=CHUNKS_SPACE, interp=INTERP, pr_tail=PR_TAIL)
    bu.save_ds(xr.Dataset(adj), out_path)
    print(f"   {name}: {(time.time() - t0) / 3600:.1f} h")

## 2. Quick checks (first 3 years): pr >= 0, tasmin < tasmax, no new NaN

In [ ]:
rows = []
for name in RUNS:
    f = OUT / f"{name}_{F1_TAG}.nc"
    if not f.exists():
        continue
    ds = xr.open_dataset(f, chunks={"time": 365}).isel(time=slice(0, 365 * 3))
    raw = run_input(name, "pr").isel(time=slice(0, 365 * 3))
    inside = raw.notnull()
    rows.append({"run": name, "variables": ", ".join(ds.data_vars),
                 "pr min": float(ds["pr"].min()),
                 "frac tasmin >= tasmax (should be 0)": float(((ds["tasmin"] >= ds["tasmax"]) & inside).sum() / inside.sum()),
                 "new NaN (should be 0)": float((ds["pr"].isnull() & inside).sum() / inside.sum())})
display(pd.DataFrame(rows).set_index("run"))

## 3. Amplification of precipitation by F1 (first 3 years, LISFLOOD domain)

Multiplicative QDM can produce very large correction factors where the model is much drier than the reference at a given
quantile. Ratio F1 / raw on wet days (raw >= 1 mm/day): maximum, 99.9th percentile, share of days amplified more than
x5, and the largest adjusted values. Large factors on many days, or adjusted values far above the raw maxima, must be
understood before using the adjusted runs (if needed: cap the factor, or more quantiles / longer calibration).

In [ ]:
LAND = bu.crop_bbox(bu.load_domain_mask(cfg, "ecearth3"), REGION)   # LISFLOOD domain on the EC-Earth3 grid (notebook 00c)
rows = []
for name in RUNS:
    f = OUT / f"{name}_{F1_TAG}.nc"
    if not f.exists():
        continue
    adj = xr.open_dataset(f, chunks={"time": 365})["pr"].isel(time=slice(0, 365 * 3))
    raw = run_input(name, "pr").isel(time=slice(0, 365 * 3))
    adj, raw = adj.where(LAND.assign_coords(lat=adj.lat, lon=adj.lon)), raw.where(LAND.assign_coords(lat=raw.lat, lon=raw.lon))
    wet = raw >= 1.0
    ratio = (adj / raw).where(wet).values.ravel()
    ratio = ratio[np.isfinite(ratio)]
    rows.append({"run": name, "wet days x pixels": ratio.size, "max factor": round(float(ratio.max()), 2),
                 "p99.9 factor": round(float(np.percentile(ratio, 99.9)), 2),
                 "frac factor > 5 (%)": round(100 * float((ratio > 5).mean()), 4),
                 "raw max (mm/day)": round(float(raw.max()), 1), "F1 max (mm/day)": round(float(adj.max()), 1)})
display(pd.DataFrame(rows).set_index("run"))